# AfriVoices submission generator (Colab fallback)

Pulls the `v3b/step_24000` checkpoint from HF -- the final checkpoint of a
long training run: warm-started from `continued_v2/step_6000`, targeting a
single continuous ~35,000-step anneal (crashed once at step 11,000 on a
dev-set zero-length-audio validation bug, resumed from that checkpoint for
the remaining 24,000 steps with validation disabled so it couldn't recur),
also using a corrected `beta_language=0.2` to push more training exposure
toward Kalenjin/Maasai (our two consistently worst-performing languages).
An interim mid-run check showed 4/6 languages improved over the previous
best, with Kalenjin/Maasai specifically better -- final numbers pending a
full evaluation of this exact checkpoint. Runs inference against the Kaggle
test set to produce `submission_step24000.csv`, then runs a CPU-only
hardware validation latency report (required by the hackathon rules).

**Before running:** Runtime > Change runtime type > GPU.

**IMPORTANT -- do not reuse old submission files for resume.** This notebook
writes to `submission_step24000.csv`, a name distinct from any earlier
`submission_step6000.csv`/`submission_step2000.csv`/`submission.csv` from
OLDER checkpoint runs, specifically so the resume-skip logic below can never
mix predictions from two different checkpoints into one file. If you want to
resume a run of *this* notebook that was interrupted, upload a partial
`submission_step24000.csv` (from this same checkpoint) via the Files pane --
never an older checkpoint's file.

In [ ]:
!pip install -q torch==2.8.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu128
!git clone -q https://github.com/facebookresearch/omnilingual-asr
!pip install -q -e "./omnilingual-asr[data]"
!pip install -q huggingface_hub kagglehub soundfile librosa pyarrow

In [ ]:
import os
from getpass import getpass

if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass("HF token: ")
os.environ["HF_HUB_DISABLE_XET"] = "1"  # Xet transfer was unreliable during data prep; plain HTTP was fine

In [ ]:
from huggingface_hub import hf_hub_download

# step_24000: final checkpoint of the long single-cycle run (see markdown
# cell above for the full lineage/rationale). LR fully re-decayed at the end
# of this segment.
CKPT_PATH = hf_hub_download(
    repo_id="Professor/afrivoices-ctc1b-checkpoints",
    repo_type="model",
    filename="v3b/step_24000/sdp_00.pt",
    token=os.environ["HF_TOKEN"],
)
print("checkpoint at:", CKPT_PATH)

In [ ]:
# If you uploaded a partial submission_step24000.csv via the Files pane (from
# an earlier interrupted run of THIS notebook, same checkpoint), the resume
# logic below will pick it up automatically. Do not rename an older
# checkpoint's submission file to this name -- that would mix predictions
# from two different checkpoints into one file.
import os
if os.path.exists("submission_step24000.csv"):
    print("submission_step24000.csv already present, will resume from it")
else:
    print("no existing submission_step24000.csv found, will start fresh")

In [ ]:
"""
Same logic as the RunPod version (training/generate_submission.py), adapted
for Colab. Batches multiple test rows per GPU call. clean_pred() below is
kept as a safety net, but should now mostly be a no-op: step_24000 (like
every checkpoint since step_2000) was trained on mojibake-corrected text
(training/patch_omnilingual_asr_mojibake.py), so it shouldn't need to
reconstruct UNK-corrupted characters at inference time the way the original
step_8000 checkpoint did.

BATCH_ROWS defaults conservatively to 4 since Colab's GPU (often a T4, 16GB)
is smaller than the A40 the RunPod run used, and test clips vary in length
up to ~39.5s -- a too-large batch risks CUDA OOM. Raise it if nvidia-smi
shows lots of headroom once running.
"""
import io, glob, csv, base64, re, unicodedata
from collections import defaultdict
import torch
import numpy as np
import soundfile as sf
import pyarrow.parquet as pq
import kagglehub

from fairseq2.models.hub import load_model
from fairseq2.data.tokenizers import load_tokenizer
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline

SR = 16000
BATCH_ROWS = 4
OUT_CSV = "submission_step24000.csv"
# Kaggle rejects the whole submission if any cell is null (empty string -> NaN
# on pandas read). Every row must have non-empty text even on failure.
EMPTY_PLACEHOLDER = "."


def clean_pred(t):
    # Safety-net only (see module docstring) -- step_24000 was trained on
    # corrected text and shouldn't produce this pattern, but keeping the
    # reconstruction logic costs nothing and guards against any residual
    # mojibake the training-time fix didn't catch (a rare, separate
    # corruption pattern was found affecting a tiny fraction of rows; see
    # training/patch_omnilingual_asr_mojibake.py docstring).
    t = unicodedata.normalize("NFKC", t)
    t = t.replace("â€™", "'")
    def repl(m, upper, lower):
        prefix = t[:m.start()]
        return upper if (m.start() == 0 or re.search(r"[.!?]\s*$", prefix)) else lower
    A_RING, A_UML, UNK = "Å", "Ä", "⁇"
    U_UP, U_LO, I_UP, I_LO = "Ũ", "ũ", "Ĩ", "ĩ"
    t2 = re.sub(A_RING + r"\s*" + UNK + r"\s*", lambda m: repl(m, U_UP, U_LO), t)
    t2 = re.sub(A_UML + r"\s*" + UNK + r"\s*", lambda m: repl(m, I_UP, I_LO), t2)
    t2 = t2.replace(UNK, "")
    return re.sub(r"\s+", " ", t2).strip()


def read_audio_bytes(raw_bytes):
    # Some test rows store WAV as base64 text instead of raw binary (same
    # quirk found in the training source data) -- try raw first, fall back.
    if not raw_bytes.startswith(b"RIFF"):
        try:
            raw_bytes = base64.b64decode(raw_bytes)
        except Exception:
            pass
    return sf.read(io.BytesIO(raw_bytes))


def prep(wav):
    if len(wav) < 2 * SR:
        wav = np.concatenate([wav, np.zeros(2 * SR - len(wav), np.float32)])
    n = 35 * SR
    return [wav] if len(wav) <= int(39.5 * SR) else [wav[i:i + n] for i in range(0, len(wav), n)]


def load_finetuned(ckpt_pt, device="cuda", dtype=torch.bfloat16):
    model = load_model("omniASR_CTC_1B_v2", device=torch.device(device), dtype=dtype)
    sd = torch.load(ckpt_pt, map_location=device)
    model.load_state_dict(sd, strict=False)
    model.eval()
    return model


def flush_batch(buffer, pipe, writer):
    all_chunks, owner = [], []
    for i, (_id, lang, chunks) in enumerate(buffer):
        for c in chunks:
            all_chunks.append(c)
            owner.append(i)
    texts = pipe.transcribe(
        [{"waveform": torch.from_numpy(c), "sample_rate": SR} for c in all_chunks],
        batch_size=len(all_chunks),
    )
    grouped = defaultdict(list)
    for idx, txt in zip(owner, texts):
        grouped[idx].append(txt)
    for i, (_id, lang, _chunks) in enumerate(buffer):
        writer.writerow({"id": _id, "language": lang, "transcription": clean_pred(" ".join(grouped.get(i, []))) or EMPTY_PLACEHOLDER})


test_path = kagglehub.dataset_download("digitalumuganda/anv-test-data-nt")
print("test set at:", test_path)

done_ids = set()
if os.path.exists(OUT_CSV):
    with open(OUT_CSV, encoding="utf-8") as f:
        done_ids = {r["id"] for r in csv.DictReader(f)}
    print(f"resuming: {len(done_ids)} ids already done")

model = load_finetuned(CKPT_PATH)
pipe = ASRInferencePipeline(model_card=None, model=model,
                             tokenizer=load_tokenizer("omniASR_tokenizer_written_v2"))

files = sorted(glob.glob(f"{test_path}/*/*/*.parquet"))
total_rows, failed_rows = len(done_ids), 0
write_header = not os.path.exists(OUT_CSV)

with open(OUT_CSV, "a", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["id", "language", "transcription"])
    if write_header:
        w.writeheader()
    buffer = []
    for i, fp in enumerate(files):
        t = pq.read_table(fp, columns=["id", "audio", "language"])
        for _id, a, lang in zip(t.column("id").to_pylist(), t.column("audio").to_pylist(), t.column("language").to_pylist()):
            if _id in done_ids:
                continue
            raw = a["bytes"] if isinstance(a, dict) else a
            try:
                wav, sr = read_audio_bytes(raw)
                if sr != SR:
                    import librosa
                    wav = librosa.resample(wav.astype(np.float32), orig_sr=sr, target_sr=SR)
                chunks = prep(wav.astype(np.float32))
            except Exception as e:
                failed_rows += 1
                print(f"WARN: failed on id={_id} lang={lang}: {e}")
                w.writerow({"id": _id, "language": lang, "transcription": EMPTY_PLACEHOLDER})
                total_rows += 1
                continue
            buffer.append((_id, lang, chunks))
            total_rows += 1
            if len(buffer) >= BATCH_ROWS:
                flush_batch(buffer, pipe, w)
                buffer = []
        if buffer:
            flush_batch(buffer, pipe, w)
            buffer = []
        f.flush()
        print(f"{i+1}/{len(files)} files done, {total_rows} rows so far ({failed_rows} failed)")

print(f"DONE: wrote {OUT_CSV}, total {total_rows} rows ({failed_rows} failed)")

In [ ]:
from google.colab import files
files.download("submission_step24000.csv")

## Hardware validation report (CPU-only latency)

Required by the hackathon rules: inference latency on CPU-only hardware (Raspberry Pi 4 / phone class, >=8GB RAM), confirming the model stays under the required <=1-2x audio duration.

**Time tradeoff:** timing the full ~30k-row test set on CPU would take too long given the current time pressure. This samples 20 clips per language (120 total) instead -- fast, and still a reasonably representative latency estimate. Re-run against the full test set later if time allows for the final, complete report.

In [ ]:
import time, random
import polars as pl

SAMPLES_PER_LANG = 20

def load_finetuned_cpu(ckpt_pt):
    m = load_model("omniASR_CTC_1B_v2", device=torch.device("cpu"), dtype=torch.float32)
    sd = torch.load(ckpt_pt, map_location="cpu")
    m.load_state_dict(sd, strict=False)
    m.eval()
    return m

by_lang = defaultdict(list)
for fp in files:
    lang = fp.replace(chr(92)+chr(92), "/").split("/")[-3]
    by_lang[lang].append(fp)

cpu_model = load_finetuned_cpu(CKPT_PATH)
cpu_pipe = ASRInferencePipeline(model_card=None, model=cpu_model,
                                 tokenizer=load_tokenizer("omniASR_tokenizer_written_v2"))

rows = []
random.seed(0)
for lang, lang_files in by_lang.items():
    sampled = 0
    shuffled_files = list(lang_files)
    random.shuffle(shuffled_files)
    for fp in shuffled_files:
        if sampled >= SAMPLES_PER_LANG:
            break
        t = pq.read_table(fp, columns=["id", "audio", "language"])
        ids = t.column("id").to_pylist()
        audios = t.column("audio").to_pylist()
        idxs = list(range(len(ids)))
        random.shuffle(idxs)
        for idx in idxs:
            if sampled >= SAMPLES_PER_LANG:
                break
            a = audios[idx]
            raw = a["bytes"] if isinstance(a, dict) else a
            try:
                wav, sr = read_audio_bytes(raw)
            except Exception:
                continue
            dur = len(wav) / sr
            t0 = time.time()
            cpu_pipe.transcribe([{"waveform": torch.from_numpy(wav.astype(np.float32)), "sample_rate": sr}], batch_size=1)
            elapsed = time.time() - t0
            rows.append({"id": ids[idx], "language": lang, "duration_s": dur, "latency_s": elapsed, "realtime_factor": elapsed / dur})
            sampled += 1
    print(lang + ": timed " + str(sampled) + " clips")

df = pl.DataFrame(rows)
df.write_csv("hardware_validation_report.csv")

rtf_col = "realtime_factor"
summary = (df.group_by("language")
             .agg(pl.col(rtf_col).mean().alias("mean_rtf"),
                  pl.col(rtf_col).max().alias("max_rtf")))
print(summary)
overall_mean = df[rtf_col].mean()
overall_max = df[rtf_col].max()
print("overall mean realtime factor: " + str(round(overall_mean, 3)))
print("overall max realtime factor: " + str(round(overall_max, 3)))
print("(must be <= 1-2x per hackathon rules)")


In [ ]:
from google.colab import files
files.download("hardware_validation_report.csv")